# TRACE Notebook

This notebook creates the tarces for all 4 mores: 
- sequential:  creates traces sequantialy 
- parallel: runs the key generation, creates the snapshot, load the snapshot and creats the ciphertext in parallel 
- sample: for more runs, for each run a new ciphertext is generated and for each of them ciphertext with fault indices are created 
- truncated: save each xs() in a separate trace call

## 1. Imports

In [ ]:
import os
import sys
import pickle
import traceback
import multiprocessing
from functools import partial

from TRACE_path_helpers import (
    get_B_csv_path,
    get_S_dir,
    get_ct_modified_path,
    get_run_ciphertext_path,
    get_sample_ct_modified_path,
    get_snapshot_path,
    get_trace_csv_path,
    get_trim_csv_path,
)
from TRACE_BS_extraction import save_S_from_sk_csv
from TRACE_ciphertext_creation import load_base_ciphertext as load_base_ciphertext_from_path
from TRACE_emulator_helpers import (
    get_label_address,
    make_disasm,
    normalize_addr,
    setup_qiling_instance,
)
from TRACE_stop_tracing import SnapshotReady, StopEmulation
from TRACE_tracing import (
    make_full_trace,
    make_snapshot_tracing,
    reset_trace_state,
    run_decapsulation_worker,
)


## 2. Global Variables

In [ ]:
md = make_disasm()

ins_trace = []
reg_trace = []

# initialization addresses
main_addr = None
trigger_high_addr = None
trigger_low_addr = None
skip = None
skip_addrs = set()
clear_bytes_addr = None

# addresses for the key generation
kem_keypair_addr = None
g_pk_addr = None
g_sk_addr = None
g_keypair_done_addr = None

# address for the ciphertext in the encapsulation/decapsulation function
g_ct_addr = None
crypto_kem_dec_addr = None

# truncated-only addresses
mul_bs_addr = None
xs_addr = None

hit_main = False
hit_kem_keypair = False
hit_crypto_kem_dec = False
hit_trigger_high = False
hit_trigger_low = False
snapshot_saved = False

trace_started = False
trace_saved = False
stop_requested = False

address_PK = None
address_SK = None
address_CT = None

instr_counter = 0
fault_index = 0
current_fault_index = 0
current_run_index = 0
current_traces_dir = None

# Whether PK/SK have already been saved
keypair_saved = False

dec_return_addr = None
global_output_dir = None
output_dir = None
output_dir_trim = None
snapshot_path = None
ct_base_path = None


## 3. Mode and Arguments

Choose one mode and run the remaining cells. The defaults mirror the Makefile.

In [ ]:
# the modes are "sequential", "parallel", "sample", and "truncated"
mode = "sequential"

elf_file = "firmware/simpleserial-frodo-CW308_STM32F4.elf"

# Sequential defaults
sequential_fault_index = 0
output_dir_sequential = "test_output_decapsulation_sequential"
output_dir_sequential_trim = "test_output_decapsulation_sequential_TRIM"

# Parallel defaults
n_parallel = 641
jobs_parallel = 10
output_dir_parallel = "test_output_decapsulation_parallel"
output_dir_parallel_trim = "test_output_decapsulation_parallel_TRIM"

# Sample defaults
num_runs_sample = 5
fault_indices_sample = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
jobs_sample = 10
output_dir_sample = "test_output_decapsulation_sample"
output_dir_sample_trim = "test_output_decapsulation_sample_TRIM"

# Truncated defaults
num_runs_truncated = 80
fault_indices_truncated = [0]
jobs_truncated = 10
output_dir_truncated = "test_output_decapsulation_truncated"
output_dir_truncated_trim = "test_output_decapsulation_truncated_TRIM"

run_snapshot = True
run_decapsulation = True
use_multiprocessing = True


## 4. Select Mode Configuration

In [ ]:
if mode == "sequential":
    fault_index = sequential_fault_index
    output_dir = output_dir_sequential
    output_dir_trim = output_dir_sequential_trim
    ct_base_path = os.path.join(output_dir, "ct_base.bin")
    snapshot_path = None
    jobs = 1
elif mode == "parallel":
    output_dir = output_dir_parallel
    output_dir_trim = output_dir_parallel_trim
    snapshot_path = get_snapshot_path(output_dir)
    jobs = min(jobs_parallel, max(n_parallel, 1))
elif mode == "sample":
    output_dir = output_dir_sample
    output_dir_trim = output_dir_sample_trim
    snapshot_path = get_snapshot_path(output_dir)
    jobs = min(jobs_sample, max(num_runs_sample * len(fault_indices_sample), 1))
elif mode == "truncated":
    output_dir = output_dir_truncated
    output_dir_trim = output_dir_truncated_trim
    snapshot_path = get_snapshot_path(output_dir)
    jobs = min(jobs_truncated, max(num_runs_truncated * len(fault_indices_truncated), 1))
else:
    raise ValueError(f"Unknown mode: {mode}")

global_output_dir = output_dir

print(f"mode = {mode}")
print(f"elf_file = {elf_file}")
print(f"output_dir = {output_dir}")
print(f"output_dir_trim = {output_dir_trim}")
if snapshot_path is not None:
    print(f"snapshot_path = {snapshot_path}")


## 5. Initialising the addresses

In [ ]:
print("Initialisation checks:")
print("-----------------------")
print("Starting script")
print(f"ELF exists? {os.path.exists(elf_file)}")

if not os.path.exists(elf_file):
    raise FileNotFoundError(elf_file)

trigger_setup_addr = normalize_addr(get_label_address(elf_file, "trigger_setup"))
init_uart_addr = normalize_addr(get_label_address(elf_file, "init_uart"))

skip = trigger_setup_addr
skip_addrs = {addr for addr in [trigger_setup_addr, init_uart_addr] if addr is not None}

clear_bytes_addr = normalize_addr(get_label_address(elf_file, "clear_bytes"))
main_addr = normalize_addr(get_label_address(elf_file, "main"))
kem_keypair_addr = normalize_addr(get_label_address(elf_file, "crypto_kem_keypair"))
trigger_high_addr = normalize_addr(get_label_address(elf_file, "trigger_high"))
crypto_kem_dec_addr = normalize_addr(get_label_address(elf_file, "crypto_kem_dec"))
trigger_low_addr = normalize_addr(get_label_address(elf_file, "trigger_low"))

g_pk_addr = get_label_address(elf_file, "g_pk")
g_sk_addr = get_label_address(elf_file, "g_sk")
g_ct_addr = get_label_address(elf_file, "g_ct")
g_keypair_done_addr = get_label_address(elf_file, "g_keypair_done")

if mode == "truncated":
    mul_bs_addr = normalize_addr(get_label_address(elf_file, "mul_bs"))
    xs_addr = normalize_addr(get_label_address(elf_file, "xs"))
    print(f"mul_bs address = {hex(mul_bs_addr) if mul_bs_addr else None}")
    print(f"xs address     = {hex(xs_addr) if xs_addr else None}")

print(f"Skip addresses = {[hex(addr) for addr in sorted(skip_addrs)]}")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir_trim, exist_ok=True)

reset_trace_state(
    globals(),
    include_main_flags=True,
    include_xs=(mode == "truncated"),
)

print(f"The {output_dir} was created")
print(f"The {output_dir_trim} was created")


## 6. Sanity check after tracing:

In [ ]:
def print_summary(mode, hit_main, hit_kem_keypair, hit_crypto_kem_dec, hit_trigger_high, hit_trigger_low,snapshot_saved=None):
    print("Summary:")
    print(f"main hit           = {hit_main}")
    print(f"keypair hit        = {hit_kem_keypair}")
    print(f"trigger_high hit   = {hit_trigger_high}")
    print(f"crypto_kem_dec hit = {hit_crypto_kem_dec}")
    print(f"trigger_low hit    = {hit_trigger_low}")

    if mode in ["parallel", "sample", "truncated"]:
        print(f"snapshot saved     = {snapshot_saved}")
    if mode not in ["sequential", "parallel", "sample", "truncated"]:
        raise ValueError(f"Unknown mode: {mode}")

## 7. Sequential Mode


In [ ]:
if mode == "sequential" and run_decapsulation:
    ql = setup_qiling_instance(elf_file, patch_uart=False, include_bitband=False)
    ql.hook_code(partial(make_full_trace, namespace=globals()))

    print("-----------------------------")
    print("Running sequential full trace...")
    try:
        ql.run()
    except StopEmulation as e:
        print(e)
    except Exception as e:
        print("Error during execution:", e)
        traceback.print_exc()

    print_summary(
        mode,
        hit_main,
        hit_kem_keypair,
        hit_crypto_kem_dec,
        hit_trigger_high,
        hit_trigger_low,
    )


## 8. Just Key Generation for Parallel, Sample, and Truncated


In [ ]:
if mode == "sequential" and run_decapsulation:
    ql = setup_qiling_instance(elf_file, patch_uart=False, include_bitband=False)
    ql.hook_code(partial(make_full_trace, namespace=globals()))

    print("-----------------------------")
    print("Running sequential full trace...")
    try:
        ql.run()
    except StopEmulation as e:
        print(e)
    except Exception as e:
        print("Error during execution:", e)
        traceback.print_exc()

    print_summary(mode, hit_main, hit_kem_keypair, hit_crypto_kem_dec, hit_trigger_high, hit_trigger_low)

if mode in {"parallel", "sample", "truncated"} and run_snapshot:
    if mode == "parallel":
        load_base_ciphertext_from_path(os.path.join(output_dir, "ct_base.bin"))
    elif mode == "sample":
        for run_index in range(num_runs_sample):
            load_base_ciphertext_from_path(get_run_ciphertext_path(output_dir, run_index))
    elif mode == "truncated":
        for run_index in range(num_runs_truncated):
            load_base_ciphertext_from_path(get_run_ciphertext_path(output_dir, run_index))

    md = make_disasm()
    ql = setup_qiling_instance(elf_file, patch_uart=False, include_bitband=False)
    ql.hook_code(partial(make_snapshot_tracing, namespace=globals()))

    print("------------------------------")
    print("Snapshot preparation: keygen + crypto_kem_dec entry")
    print("------------------------------")
    try:
        ql.run()
    except SnapshotReady as e:
        print(e)
    except StopEmulation as e:
        print(e)
    except Exception as e:
        print(f"Error during snapshot run: {e}")
        traceback.print_exc()

    print_summary(mode, hit_main, hit_kem_keypair, hit_crypto_kem_dec, hit_trigger_high, hit_trigger_low, snapshot_saved,)

    if mode == "truncated":
        sk_path = os.path.join(output_dir, "sk.bin")
        if os.path.exists(sk_path):
            with open(sk_path, "rb") as f:
                sk = f.read()
            save_S_from_sk_csv(sk, os.path.join(get_S_dir(output_dir), "S.csv"))
        else:
            print(f"Missing sk.bin, cannot save S.csv: {sk_path}")


## 8. Decapsulation for Parallel, Sample, Truncated

In [ ]:
if mode in {"parallel", "sample", "truncated"} and run_decapsulation:
    if snapshot_path is None or not os.path.exists(snapshot_path):
        raise FileNotFoundError(f"Missing snapshot: {snapshot_path}")

    if mode == "parallel":
        worker_args = [
            (
                i,
                snapshot_path,
                elf_file,
                output_dir,
                output_dir_trim,
                trigger_high_addr,
                trigger_low_addr,
                tuple(skip_addrs),
                g_ct_addr,
                clear_bytes_addr,
            )
            for i in range(n_parallel)
        ]
    elif mode == "sample":
        worker_args = [
            (
                run_index,
                fault_index,
                snapshot_path,
                elf_file,
                output_dir,
                output_dir_trim,
                trigger_high_addr,
                trigger_low_addr,
                tuple(skip_addrs),
                g_ct_addr,
                clear_bytes_addr,
            )
            for run_index in range(num_runs_sample)
            for fault_index in fault_indices_sample
        ]
    elif mode == "truncated":
        worker_args = [
            (
                run_index,
                fault_index,
                snapshot_path,
                elf_file,
                output_dir,
                output_dir_trim,
                trigger_high_addr,
                trigger_low_addr,
                tuple(skip_addrs),
                g_ct_addr,
                clear_bytes_addr,
                mul_bs_addr,
                xs_addr,
            )
            for run_index in range(num_runs_truncated)
            for fault_index in fault_indices_truncated
        ]

    print(f"Starting {len(worker_args)} decapsulation worker task(s) in mode={mode}")

    if use_multiprocessing and len(worker_args) > 1:
        ctx = multiprocessing.get_context("spawn")
        with ctx.Pool(processes=jobs) as pool:
            pool.starmap(run_decapsulation_worker, [(args, mode) for args in worker_args])
    else:
        for args in worker_args:
            run_decapsulation_worker(args, mode)

    print("All traces have been collected successfully")
